In [1]:
PD_SUBJECTS = [
    "sub-pd11","sub-pd12","sub-pd13","sub-pd14",
    "sub-pd16","sub-pd17","sub-pd19","sub-pd22",
    "sub-pd23","sub-pd26","sub-pd28","sub-pd3",
    "sub-pd5","sub-pd6","sub-pd9"
]

HC_SUBJECTS = [
    "sub-hc1","sub-hc10","sub-hc18","sub-hc2",
    "sub-hc20","sub-hc21","sub-hc24","sub-hc25",
    "sub-hc29","sub-hc30","sub-hc32","sub-hc33",
    "sub-hc4","sub-hc7","sub-hc8","sub-hc31"
]


In [2]:
import os
import numpy as np
import mne

PD_DIR = r"C:\Users\enneh\eeg_band_data-seson"
HC_DIR = r"C:\Users\enneh\eeg_band_data-hc"
SAVE_DIR = r"C:\Users\enneh\dl_dataset"
os.makedirs(SAVE_DIR, exist_ok=True)

SFREQ = 512

freq_bands = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta":  (13, 30),
    "gamma": (30, 48)
}


In [6]:
def build_band_dataset(subject_list, data_dir, label, band_range):
    X_list, y_list, subj_list = [], [], []

    for subj in subject_list:
        file_path = os.path.join(data_dir, f"{subj}_X.npy")
        if not os.path.exists(file_path):
            print(f"⚠ Missing file: {file_path}")
            continue

        eeg = np.load(file_path)  # shape: (epochs, channels, times)

        # ---- Handle >3D EEG data ----
        if eeg.ndim > 3:
            n1, n2, n3, n4 = eeg.shape
            eeg_reshaped = eeg.reshape(n1 * n2, n3, n4)
            band_eeg = mne.filter.filter_data(
                eeg_reshaped,
                sfreq=SFREQ,
                l_freq=band_range[0],
                h_freq=band_range[1],
                verbose=False
            )
            band_eeg = band_eeg.reshape(n1, n2, n3, n4)
            band_eeg = band_eeg.reshape(-1, n3, n4)
        else:
            band_eeg = mne.filter.filter_data(
                eeg,
                sfreq=SFREQ,
                l_freq=band_range[0],
                h_freq=band_range[1],
                verbose=False
            )

        X_list.append(band_eeg)
        y_list.append(np.full(band_eeg.shape[0], label))
        subj_list.append(np.array([subj] * band_eeg.shape[0]))

    return X_list, y_list, subj_list

In [ ]:
for band_name, band_range in freq_bands.items():
    print(f"\nCreating {band_name} dataset...")

    X_pd, y_pd, s_pd = build_band_dataset(PD_SUBJECTS, PD_DIR, 1, band_range)
    X_hc, y_hc, s_hc = build_band_dataset(HC_SUBJECTS, HC_DIR, 0, band_range)

    if len(X_pd) == 0 or len(X_hc) == 0:
        raise RuntimeError(f"No data found for {band_name}")

    X = np.concatenate(X_pd + X_hc, axis=0)
    y = np.concatenate(y_pd + y_hc, axis=0)
    subjects = np.concatenate(s_pd + s_hc, axis=0)

    # Optional normalization
    X = (X - X.mean(axis=-1, keepdims=True)) / X.std(axis=-1, keepdims=True)

    # Save datasets
    np.save(os.path.join(SAVE_DIR, f"{band_name}_X.npy"), X)
    np.save(os.path.join(SAVE_DIR, f"{band_name}_y.npy"), y)
    np.save(os.path.join(SAVE_DIR, f"{band_name}_subjects.npy"), subjects)

    print(f"{band_name}: X{X.shape}, y{y.shape}, subjects{subjects.shape}")


Creating delta dataset...


C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (1691) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (1691) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (1691) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (1691) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\L

delta: X(27900, 32, 512), y(27900,), subjects(27900,)

Creating theta dataset...


C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (845) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (845) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (845) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (845) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local

theta: X(27900, 32, 512), y(27900,), subjects(27900,)

Creating alpha dataset...


C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (845) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (845) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (845) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (845) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local

alpha: X(27900, 32, 512), y(27900,), subjects(27900,)

Creating beta dataset...


C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (521) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (521) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (521) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local\Temp\ipykernel_26896\732143033.py:16: RuntimeWarning: filter_length (521) is longer than the signal (512), distortion is likely. Reduce filter length or filter a longer signal.
  band_eeg = mne.filter.filter_data(
C:\Users\enneh\AppData\Local

beta: X(27900, 32, 512), y(27900,), subjects(27900,)

Creating gamma dataset...
